**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/nakagawa-2026"], check=True)
    os.chdir("/content/encore/entries/nakagawa-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Nakagawa et al. (2026) from dandiset 001968

Paper. Nakagawa T, Xie JL, Park K, et al., Igarashi KM (2026). Early dopamine disruption in
the entorhinal cortex of a knock-in model of Alzheimer's disease. *Nature Neuroscience*.
[doi:10.1038/s41593-026-02260-w](https://doi.org/10.1038/s41593-026-02260-w). The claims file was
written from the version of record (Europe PMC, PMC13246448) with its Extended Data legends and
the supplement (Supplementary Figs. 1 to 11, Tables 1 to 4), and from the Methods of ref. 16 (Lee
et al. 2021) for the steps the paper describes "as previously described".

Dandiset. [DANDI:001968](https://dandiarchive.org/dandiset/001968/0.260914.1316), version
0.260914.1316. The paper cites no DANDI version (its data statement points to the lab's web page,
which holds only a hippocampal spike set of an earlier paper, three linear-track MATLAB scripts and
a headplate file). Of the two published versions, 0.260913.1518 holds a single unit file and
0.260914.1316 all 3855 files; the latter is the most recent and is pinned. The draft has the same
asset count and size.

What the dandiset holds. 3813 unit files (one sorted LEC unit each: spike times, a source mean
rate, and a source file name carrying strain label, date, odor set, an RS or FS class label and
tetrode) from 27 tetrode mice, and 41 photometry sessions from 10 mice (one z-scored trace per
trial with its odor and action codes, and the source's per-odor block means). There are no trial,
odor or lick times for the unit sessions, no waveforms, no histology, and no behaviour of mice other
than the photometry mice. No file states genotype, age group or treatment; these come from the
paper's Supplementary Table 2 and Supplementary Fig. 2a (link claims below).

| Claim | Tested here |
|---|---|
| link-unit-mice, link-photometry-mice (added by the author) | tested |
| units-recorded, ed4c-higher-firing-rates | tested |
| ed4a-spike-type-distribution, ed4b-spike-width | blocked: no waveforms (a two-class variant is shown for ed4a) |
| fig2d, ed5, fig2j, fig2g, fig2f, fig2i, fig2k, fig2l, ed8, ed9, ed7, fig6h, fig6i (13 claims on cue responses, PCA, decoding) | blocked: no event times for the unit sessions |
| fig3b, fig3d, ed10, ed1 (histology) | blocked: not in the dandiset |
| fig4b, fig4c, fig4d, fig4e, fig4f, figs7, figs6d (photometry) | tested |
| fig1d-young-new-association-impaired, fig1e-young-correct-sessions | tested on the 10 photometry mice |
| fig1d-prelearned-retrieval-intact, fig1g, fig1h, figs1c, figs1f | blocked: behaviour of old mice and of prelearning not in the dandiset |
| fig5c, fig5d, figs9a, figs9b (optogenetics); fig6b, fig6c, fig6k, figs11 (L-DOPA behaviour) | blocked: not in the dandiset |

The helper module `lecda.py` loads the files and holds the measures; its docstring says where each
parameter comes from. Two choices rest on assumptions checked in the data check: the photometry
traces are sampled at 1 kHz with the cue at sample 1000, and the odor codes 1 to 4 are Odor-A,
Odor-B, Odor-1 and Odor-2. A third, the unit of the photometry statistics, is set out before the photometry
claims: the mean over a mouse's sessions stands in for the paper's hemispheres.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import lecda

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
records = lecda.load_all()
units = lecda.unit_table(records)
sessions = lecda.sessions_table(records)
print(f"read {len(records)} files of dandiset {lecda.DANDISET_ID} version {lecda.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Versions of the dandiset:")
print(lecda.version_summary().to_string(index=False))
early = lecda.version_paths("0.260913.1518")
print("\nThe earlier published version holds:", early)
kinds = pd.DataFrame([{"subject": r["subject"], "kind": r["kind"], "session": r["session_id"]} for r in records])
print("\nFiles by kind:", kinds.kind.value_counts().to_dict())
print("Subjects:", kinds.subject.nunique())
print("\nGroup columns present (per file kind):")
for k in kinds.kind.unique():
    print(" ", k, sorted({tuple(r["groups"]) for r in records if r["kind"] == k}))
print("\nFiles with a trials table or intervals:", sum("intervals" in r["groups"] for r in records))
print("Unit-file notes, first file:", next(r["notes"] for r in records if r["kind"] == "unit")[:400])
empty = [r for r in records if r["kind"] == "empty"]
print("\nFiles with no units and no photometry:", [(r["path"], r["source"]) for r in empty])

The pinned version, 0.260914.1316, is the most recent and the only complete one; the earlier
0.260913.1518 holds a single unit file. Unit files hold no trial table, no intervals and no
stimulus or behaviour series, so no unit can be aligned to an odor. One unit file
(mouse02491, 2025-04-08, TT12 cluster 01) holds no units table at all; its notes name the source
MAT file only.

In [ ]:
print("Every group and dataset of three files (NWB specifications left out):")
assets = lecda.list_assets().set_index("path")
pick = {"unit file with scratch": next(r for r in records if r["kind"] == "unit" and "scratch" in r["groups"]),
        "unit file without scratch": next(r for r in records if r["kind"] == "unit" and "scratch" not in r["groups"]),
        "photometry file": next(r for r in records if r["kind"] == "photometry")}
for what, r in pick.items():
    print(f"\n{what}: {r['path']}")
    for name, shape, kind in lecda.file_tree(assets.loc[r["path"], "asset_id"]):
        print("   ", name, "" if shape is None else shape, kind)
print("\nUnits-table columns over all unit files:", sorted({tuple(r["unit_columns"]) for r in records if r["kind"] == "unit"}))
print("Photometry notes, first file:", next(r["notes"] for r in records if r["kind"] == "photometry"))

No unit file holds waveforms, electrode or channel tables, trial tables, events, stimulus or
acquisition series: the units table has the columns listed, and `scratch` holds only a copy of the
source spike times. The photometry files hold the trial table of z-scored traces with odor and action
codes, a sample axis and the four block tables, and no fiber, hemisphere or location metadata; the
notes say that the sampling rate, alignment timing, brain region and the codes' meanings are
unknown.

In [ ]:
subj = []
for m, d in kinds.groupby("subject"):
    rs = [r for r in records if r["subject"] == m]
    subj.append({"mouse": m, "paper group": lecda.group_of(m), "files": len(d),
                 "kinds": "/".join(sorted(d.kind.unique())), "sessions": d.session.nunique(),
                 "sex (file)": "/".join(sorted({str(r["sex"]) for r in rs})),
                 "sex (Supp. Table 2)": lecda.PAPER_SEX.get(m),
                 "genotype field": "/".join(sorted({str(r["genotype"]) for r in rs})),
                 "date of birth": "/".join(sorted({str(r["dob"]) for r in rs}))})
subj = pd.DataFrame(subj)
print(subj.to_string(index=False))
missing = [m for g in lecda.PAPER_GROUPS.values() for m in g if m not in set(subj.mouse)]
print("\nAnimals of the paper's tetrode and photometry rows not in the dandiset:", missing)
print("Sex disagreements:", subj[(subj["sex (file)"] != subj["sex (Supp. Table 2)"])][["mouse", "sex (file)", "sex (Supp. Table 2)"]].to_dict("records"))

No file states a genotype, except the duplicated mouse00375 files, which give "Sim1". The groups
are taken from the paper (helper, `PAPER_GROUPS`). Supplementary Table 2 and Supplementary Fig. 2
disagree on which five APP-KI tetrode mice are young; the dates of birth in the files agree with
Supplementary Fig. 2 (ages below).

In [ ]:
print("Unit files by strain label in the source file name and the paper's group:")
print(pd.crosstab(units.group, units.label).to_string())
print("\nCell class label by group:")
print(pd.crosstab(units.group, units.cls).to_string())
ages = units.groupby(["group", "mouse"]).agg(label=("label", "first"), age_min=("age_months", "min"),
                                             age_max=("age_months", "max"), sessions=("date", "nunique"),
                                             units=("path", "size")).round(1)
print("\nAge at recording (months, from date of birth and the date in the file name):")
print(ages.to_string())
print("\nPaper: young 3 to 6 months, old 7 to 11 months (Methods, Animals).")

The file names label mice 01398 and 01431 "AD", where Supplementary Table 2 and Supplementary
Fig. 2 list them as young WT. Mouse 00789's date of birth puts it at 1.3 to 1.6 months, below
the paper's young range. The other ages agree with the groups of Supplementary Fig. 2. The
label "NLGF" is used for both the L-DOPA and the saline mice, so the treatment is not in the
dandiset; it comes from Supplementary Table 2.

In [ ]:
dup = lecda.duplicate_units(units)
print("Unit files whose spike train (count, first and last spike) also occurs in another file:", len(dup))
print("  of which '_Sum' files (genotype field 'Sim1'):", int(dup.sum_file.sum()), "of", int(units.sum_file.sum()), "'_Sum' files")
print("  mice:", sorted(dup.mouse.unique()))
pairs = dup.groupby("key").agg(files=("source", list))
print("Example pairs:")
for f in pairs.files.head(3):
    print("   ", f)
same = dup.groupby("key").agg(n=("path", "size"))
print("Groups by size:", same.n.value_counts().to_dict())
units = units[~units.sum_file].copy()
print("\nAfter dropping the '_Sum' copies:", len(units), "unit files")
print("Spike times sorted in every file:", bool(units.sorted.all()))
print("Spike time range (s): first", round(units.t_first.min(), 1), "to", round(units.t_first.max(), 1),
      "; last", round(units.t_last.min(), 1), "to", round(units.t_last.max(), 1))
print("Span of each unit (first to last spike), minutes:", (units.t_last - units.t_first).div(60).describe().round(1).to_dict())
print("Source meanFR against spikes / span: median ratio", round((units.meanFR / units.rate_span).median(), 4),
      "; Pearson r", round(np.corrcoef(units.meanFR, units.rate_span)[0, 1], 4))
print("Units with at most 100 spikes:", int((units.n_spikes <= 100).sum()))
print("Odor-set field values (number of unit files):", units.odors.value_counts().head(12).to_dict())

The 100 "_Sum" files of mouse00375 (genotype field "Sim1", one file per cell, odor sets written
without A and B) repeat the spike trains of its 99 regular files, one of which has two copies.
They are dropped and each recording counted once. The source `meanFR` equals the spike count over
the first-to-last spike span (median ratio near 1), so it is a whole-recording rate. The odor-set
field sometimes doubles the novel letters (ABCCDD against ABCD); the files do not say what this
means.

In [ ]:
ph = pd.DataFrame([{"session": f"{s['mouse']}_{s['date']}", "mouse": s["mouse"], "group": s["group"],
                    "trials": len(s["odor"]), "samples": s["X"].shape[1],
                    **{f"odor {c}": int((s["odor"] == c).sum()) for c in (1, 2, 3, 4)}} for s in sessions])
print("Photometry sessions per group:", ph.groupby("group").session.nunique().to_dict(),
      "; mice:", ph.groupby("group").mouse.nunique().to_dict())
print(ph.to_string(index=False))
print("\nTrials per session:", ph.trials.min(), "to", ph.trials.max(), "(paper: about 20 A/B trials, then 160 AB12 trials)")
allo = np.concatenate([s["odor"] for s in sessions]); alla = np.concatenate([s["action"] for s in sessions])
print("\nOdor code by action code, all sessions:")
print(pd.crosstab(pd.Series(allo, name="odor"), pd.Series(alla, name="action")).to_string())
print("\nShare of each odor code in trials:", {c: round(float((allo == c).mean()), 3) for c in (1, 2, 3, 4)},
      "(paper: A 20%, B 20%, 1 30%, 2 30% of AB12 trials)")
first20 = np.concatenate([s["odor"][:20] for s in sessions])
print("Share of codes 3 and 4 in the first 20 trials of each session:", round(float(np.isin(first20, [3, 4]).mean()), 3))

Action codes 1 and 2 occur only with odor codes 1 and 3, and 3 and 4 only with odor codes 2 and
4, so 1 and 3 are the rewarded odors and 2 and 4 the punished ones; codes 1 and 4 (hit, correct
rejection) dominate. Codes 3 and 4 are present from the first trials, so the files hold no
separate A/B-only block, and sessions run 99 to 269 trials, not 180.

In [ ]:
X0 = np.vstack([s["X"] for s in sessions]).astype(float)
base = X0[:, :1000]
print("Per trial, samples 0 to 999: |mean| max", round(float(np.abs(base.mean(1)).max()), 4),
      "; SD median", round(float(np.median(base.std(1))), 4), "min", round(float(base.std(1).min()), 4),
      "max", round(float(base.std(1).max()), 4))
for a, b in ((0, 500), (0, 2000), (1000, 2000)):
    print(f"Per trial, samples {a} to {b - 1}: |mean| median", round(float(np.median(np.abs(X0[:, a:b].mean(1)))), 3),
          "; SD median", round(float(np.median(X0[:, a:b].std(1))), 3))
seg = base - base.mean(1, keepdims=True)
spec = np.abs(np.fft.rfft(seg, axis=1)).mean(0)
freq = np.fft.rfftfreq(1000, 1 / lecda.FS_HZ)
top = np.argsort(spec[5:])[::-1][:6] + 5
print("\nLargest peaks of the precue spectrum, if 1000 samples are 1 s (Hz):", sorted(freq[top].tolist()))
print("The same peaks if the rate were 250 Hz:", sorted((freq[top] / 4).tolist()))

Every trial column has mean 0 and SD 1 over its first 1000 samples and over no other span
tested: the columns are z-scored to a 1000-sample precue period. The paper's photometry is
z-scored to the 1 s before cue (claims file, Shared method), so 1000 samples are read as 1 s,
with the cue at sample 1000 and the trial spanning -1 to 5 s. The noise peaks at 120 Hz and its
harmonics under this rate, as mains interference in a US laboratory would. This supports the
reading but does not prove it; it is an assumption of every photometry claim.

In [ ]:
rows = []
for s in sessions:
    for t, code in zip("ABCD", (1, 2, 3, 4)):
        idx = np.flatnonzero(s["odor"] == code)
        tab = s["tables"][t]
        starts = lecda.block_starts(len(idx))
        for k in range(tab.shape[0]):
            cand = range(0, len(idx) - 9)
            r = [np.corrcoef(s["X"][idx[c:c + 10]].mean(0), tab[k])[0, 1] for c in cand]
            best = int(np.argmax(r))
            rows.append({"session": f"{s['mouse']}_{s['date']}", "table": t, "column": k + 1,
                         "best_start": best, "r_best": r[best], "rule_start": starts[k],
                         "r_rule": r[starts[k]]})
tb = pd.DataFrame(rows)
print("Table columns compared:", len(tb), "; columns per table:", tb.groupby(["session", "table"]).size().unique().tolist())
print("Correlation of each table column with the mean of 10 trials of its odor code, at the best start: median",
      round(tb.r_best.median(), 4), "min", round(tb.r_best.min(), 4))
print("Best start equal to the rule round(k (n - 10) / 4):", round(float((tb.best_start == tb.rule_start).mean()), 3),
      "; within 1 trial:", round(float((abs(tb.best_start - tb.rule_start) <= 1).mean()), 3))
print("Correlation at the rule's start: median", round(tb.r_rule.median(), 4), "; 5th percentile", round(tb.r_rule.quantile(0.05), 4))
cross = []
for s in sessions[:5]:
    for t, code in zip("ABCD", (1, 2, 3, 4)):
        for other in (1, 2, 3, 4):
            idx = np.flatnonzero(s["odor"] == other)
            cross.append({"table": t, "code": other, "r": np.corrcoef(s["X"][idx[:10]].mean(0), s["tables"][t][0])[0, 1]})
print("\nT1 column of each table against the first 10 trials of each code (first 5 sessions), mean r:")
print(pd.DataFrame(cross).pivot_table(index="table", columns="code", values="r").round(2).to_string())

Each table `zscore_A` to `zscore_D` holds five columns, each the mean of 10 trials of odor code 1
to 4 (correlation near 1 with the unsmoothed trial means, the residual being the tables' own
smoothing). Their starts follow round(k (n - 10) / 4): T1 is the first 10 trials of an odor, T5
the last 10, T3 the middle 10, T2 and T4 between. This links codes 1 to 4 to the source's names
A, B, C, D (Odor-A, Odor-B, Odor-1, Odor-2) and fixes T1 to T5 as the dataset's authors computed
them.

In [ ]:
rng = np.random.default_rng(0)
rows = []
for s in sessions:
    v = lecda.window_mean(s["Xs"], "G1")
    rew = np.isin(s["odor"], [1, 3])
    def contrast(lab):
        return v[lab].mean() - v[~lab].mean()
    rows.append({"session": f"{s['mouse']}_{s['date']}", "true": contrast(rew),
                 "shift +1": contrast(np.roll(rew, 1)), "shift -1": contrast(np.roll(rew, -1)),
                 "precue window true": lecda.window_mean(s["Xs"], (-0.9, -0.1))[rew].mean() - lecda.window_mean(s["Xs"], (-0.9, -0.1))[~rew].mean()})
ck = pd.DataFrame(rows)
print("Rewarded minus punished trials, mean z in 1 to 4 s after cue, per session:")
print(ck.describe().round(3).to_string())
print("Sessions with true contrast above both shifted ones:", int(((ck.true > ck["shift +1"]) & (ck.true > ck["shift -1"])).sum()), "of", len(ck))
print("Wilcoxon true against shift +1:", stats.wilcoxon(ck.true, ck["shift +1"]))

The trials' labels and signals share one clock: the rewarded minus punished contrast after the
cue is large with the true labels and near 0 with the labels shifted by one trial, and near 0
before the cue. For the units no such check is possible, as no event times exist; the spike times
run from an unstated origin.

In [ ]:
X0 = np.vstack([s["X"] for s in sessions]).astype(float)
codes = np.concatenate([s["odor"] for s in sessions])
m_all = X0.mean(0)
step = np.abs(np.diff(m_all))
top = np.argsort(step)[::-1][:5]
print("Largest steps between consecutive samples of the mean of all", len(X0), "trials (sample, |step| z):",
      [(int(i + 1), round(float(step[i]), 3)) for i in top], "; median step", round(float(np.median(step)), 4))
for c in (1, 2, 3, 4):
    m = X0[codes == c].mean(0); st = np.abs(np.diff(m))
    print(f"  odor code {c}: largest step at sample {int(np.argmax(st) + 1)}, step 999-1004 max {st[999:1004].max():.3f}, "
          f"median step {np.median(st):.4f}")
loc = []
for s in sessions:
    st = np.abs(np.diff(s["X"].astype(float).mean(0)))
    loc.append(int(np.argmax(st[500:1500]) + 501))
loc = np.array(loc)
print("Per session, sample of the largest step of the mean trace among samples 500 to 1500: within 1000 to 1004 in",
      int(((loc >= 1000) & (loc <= 1004)).sum()), "of", len(loc), "; values", sorted(loc.tolist()))
t = (np.arange(lecda.N_SAMPLES) - lecda.CUE_SAMPLE) / lecda.FS_HZ
b = 50
print("\nMean z per 50-sample bin, -0.3 to 1.0 s, by odor code (rewarded 1 and 3, punished 2 and 4):")
tab = pd.DataFrame({f"code {c}": [X0[codes == c][:, k:k + b].mean() for k in range(700, 2000, b)] for c in (1, 2, 3, 4)},
                   index=[round(float(t[k]), 2) for k in range(700, 2000, b)]).round(3)
print(tab.T.to_string())
fig, ax = plt.subplots(figsize=(7, 3))
sel = slice(500, 3000)
for c, col in zip((1, 2, 3, 4), ("tab:blue", "tab:orange", "tab:green", "tab:red")):
    ax.plot(t[sel], X0[codes == c][:, sel].mean(0), color=col, lw=1.5, label=f"code {c} ({lecda.ODOR_NAMES[c]})")
ax.axvline(0, color="0.5", lw=0.8, ls="--")
ax.set_xlabel("time from sample 1000 (s, at 1 kHz)"); ax.set_ylabel("mean z (unsmoothed)"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()
lo, hi = min(X0[codes == c][:, sel].mean(0).min() for c in (1, 2, 3, 4)), max(X0[codes == c][:, sel].mean(0).max() for c in (1, 2, 3, 4))
print(f"Plotted range of the means: {lo:.2f} to {hi:.2f} z; all points inside the axes.")

A check of the cue time that does not depend on the z-scoring: the mean of all trials has a sharp
step at samples 1001 to 1002, the largest step between consecutive samples anywhere in the trace
(0.072 z, against a median step of 0.0055), and it is the largest step in each of the four odor
codes alone. An event-locked artefact at the cue (such as the odor valve) would give this. In
single sessions the step is hidden by noise: it is the largest step between samples 500 and 1500
in 7 of 41 sessions. The rewarded codes (1, 3) rise above the punished ones from about 0.45 to 0.5 s
after sample 1000. With the precue z-scoring and the mains peaks, this is consistent with the cue
at sample 1000 and a rate of 1 kHz; it does not prove the rate.

## link-unit-mice

> The dandiset's tetrode subjects can be matched to the paper's animal lists (Supplementary Table
> 2 and Supplementary Fig. 2), and the matched records agree on strain, sex and the young mice's
> principal cell counts.

In [ ]:
tet = units.groupby("mouse").agg(group=("group", "first"), label=("label", "first"), sex=("sex", "first"),
                                 units=("path", "size"),
                                 principal=("cls", lambda c: int((c == "RS").sum()))).reset_index()
tet["label strain"] = tet.label.map({"WT": "WT", "AD": "APP-KI", "NLGF": "APP-KI"})
tet["paper strain"] = tet.group.map(lambda g: "WT" if "WT" in g else "APP-KI")
tet["sex (paper)"] = tet.mouse.map(lecda.PAPER_SEX)
pr100 = units[(units.cls == "RS") & (units.n_spikes > 100)].groupby("mouse").size()
tet["principal >100 spikes"] = tet.mouse.map(pr100).fillna(0).astype(int)
tet["Supp. Fig. 2 count"] = tet.mouse.map(lecda.PAPER_CELLS_PER_MOUSE)
paper_tet = [m for g in ("young WT", "young APP-KI", "old WT", "old APP-KI", "L-DOPA", "saline") for m in lecda.PAPER_GROUPS[g]]
print(tet.to_string(index=False))
print("\nTetrode mice in the paper:", len(paper_tet), "; matched in the dandiset:", int(tet.mouse.isin(paper_tet).sum()),
      "; dandiset tetrode mice not in the paper:", sorted(set(tet.mouse) - set(paper_tet)))
print("Strain label agrees with the paper's group:", int((tet["label strain"] == tet["paper strain"]).sum()), "of", len(tet),
      "; disagree:", tet[tet["label strain"] != tet["paper strain"]].mouse.tolist())
print("Sex agrees:", int((tet.sex == tet["sex (paper)"]).sum()), "of", len(tet),
      "; disagree:", tet[tet.sex != tet["sex (paper)"]][["mouse", "sex", "sex (paper)"]].to_dict("records"))
y = tet.dropna(subset=["Supp. Fig. 2 count"])
within = (abs(y["principal >100 spikes"] - y["Supp. Fig. 2 count"]) <= 0.1 * y["Supp. Fig. 2 count"])
print("Young mice whose principal count is within 10% of Supplementary Fig. 2:", int(within.sum()), "of", len(y),
      y[within].mouse.tolist())

**Result: partially reproduced.** All 27 tetrode mice of the paper are in the dandiset, and it holds no other tetrode mouse. The strain label in the file names agrees with the paper for 25 of 27; mice 01398 and 01431, young WT in Supplementary Table 2 and Supplementary Fig. 2a, are labelled AD. Sex agrees for 25 of 27 (00703 and 00789 differ). Principal cell counts are within 10% of Supplementary Fig. 2a for 7 of 10 young mice; 00375 (72 against 48), 00390 (151 against 130) and 01398 (203 against 112) hold more.

## link-photometry-mice

> The dandiset's photometry subjects can be matched to the photometry rows of Supplementary Table
> 2, and the matched records agree on sex and on the number of recordings.

In [ ]:
pm = ph.groupby(["mouse", "group"]).agg(sessions=("session", "nunique")).reset_index()
pm["sex (file)"] = pm.mouse.map(subj.set_index("mouse")["sex (file)"])
pm["sex (paper)"] = pm.mouse.map(lecda.PAPER_SEX)
pm["hemispheres (paper)"] = pm.mouse.map(lecda.PAPER_HEMISPHERES)
print(pm.to_string(index=False))
paper_ph = lecda.PAPER_GROUPS["DAT-Cre"] + lecda.PAPER_GROUPS["APP-KI x DAT-Cre"]
print("\nPhotometry mice in Supplementary Table 2:", len(paper_ph), "; in the dandiset:", int(pd.Series(paper_ph).isin(pm.mouse).sum()),
      "; missing:", sorted(set(paper_ph) - set(pm.mouse)))
print("Sex agrees:", int((pm["sex (file)"] == pm["sex (paper)"]).sum()), "of", len(pm))
hemi = {g: sum(lecda.PAPER_HEMISPHERES[m] for m in lecda.PAPER_GROUPS[g]) for g in ("DAT-Cre", "APP-KI x DAT-Cre")}
print("Hemispheres in Supplementary Table 2:", hemi, "(Fig. 4 legends: 10 and 8)")
print("Hemispheres of the mice in the dandiset:", pm.groupby("group")["hemispheres (paper)"].sum().to_dict())
print("Recordings per file: one signal column per trial; files per group:", pm.groupby("group").sessions.sum().to_dict())

**Result: partially reproduced.** 10 of the 11 photometry mice of Supplementary Table 2 are in the dandiset; 00163 (DAT-Cre, one hemisphere) is missing. Sex agrees for 9 of 10 (00265 is F in the file and M in the table). The table implies 10 and 8 hemispheres, as the Fig. 4 legends state; the mice present account for 9 and 8. The files hold one trace per trial and session, 11 DAT-Cre and 30 APP-KI x DAT-Cre sessions, so hemispheres cannot be told apart.

## units-recorded

> In LEC layer 2/3 of young mice, 479 neurons were recorded from 5 WT mice and 846 from 5 APP-KI
> mice.

Groups as in Supplementary Fig. 2 (reading P) and, as a variant, with mice 01398 and 01431 moved to
APP-KI as the file names label them (reading F). Principal cells are those labelled RS with more
than 100 spikes (Methods, Spike sorting), as no waveforms are in the files.

In [ ]:
def young_groups(reading):
    g = units.group.copy()
    if reading == "F":
        g[units.mouse.isin(["01398", "01431"])] = "young APP-KI"
    return g
for reading in ("P", "F"):
    g = young_groups(reading)
    d = units[g.isin(["young WT", "young APP-KI"])].assign(g=g)
    tab = d.groupby("g").agg(mice=("mouse", "nunique"), units=("path", "size"),
                             RS=("cls", lambda c: int((c == "RS").sum())), FS=("cls", lambda c: int((c == "FS").sum())),
                             principal=("n_spikes", lambda n: int(((d.loc[n.index, "cls"] == "RS") & (n > 100)).sum())))
    tab["paper units"] = tab.index.map({"young WT": 479, "young APP-KI": 846})
    tab["paper principal"] = tab.index.map({"young WT": 423, "young APP-KI": 636})
    tab["paper interneurons"] = tab.index.map({"young WT": 56, "young APP-KI": 210})
    tab["units within 10%"] = abs(tab.units - tab["paper units"]) <= 0.1 * tab["paper units"]
    print(f"Reading {reading}:"); print(tab.to_string()); print()
odd = units[units.odors.str.len() == 6]
print("Units in sessions whose odor set doubles the novel letters, young mice (reading P):",
      odd[odd.group.isin(["young WT", "young APP-KI"])].groupby("group").size().to_dict())
for g in ("old WT", "old APP-KI", "L-DOPA", "saline"):
    d = units[units.group == g]
    print(f"{g}: {len(d)} units, {int(((d.cls == 'RS') & (d.n_spikes > 100)).sum())} principal",
          {"old WT": "(paper 671 cells)", "old APP-KI": "(paper 618)", "L-DOPA": "(paper 144)", "saline": "(paper 132)"}[g])

In [ ]:
y = units[units.group.isin(["young WT", "young APP-KI"]) & (units.cls == "RS") & (units.n_spikes > 100)]
rows = []
for (g, m), d in y.groupby(["group", "mouse"]):
    per = d.groupby("date").agg(n=("path", "size"), odors=("odors", "first")).sort_index()
    cum = per.n.cumsum().values
    paper = lecda.PAPER_CELLS_PER_MOUSE[m]
    k = int(np.argmin(abs(cum - paper)))
    four = int(per[per.odors.str.len() == 4].n.sum())
    rows.append({"group": g, "mouse": m, "sessions": len(per), "principal": int(cum[-1]), "Supp. Fig. 2a": paper,
                 "in 4-letter sessions": four, "in doubled-letter sessions": int(cum[-1]) - four,
                 "first k sessions closest": k + 1, "count in first k": int(cum[k]),
                 "per session (date order)": per.n.tolist(), "odor sets": "/".join(per.odors)})
sel = pd.DataFrame(rows)
print(sel.drop(columns="odor sets").to_string(index=False))
print("\nOdor sets per session (date order):")
for r in rows:
    print(" ", r["mouse"], r["odor sets"])
for rule, col in (("all sessions", "principal"), ("4-letter sessions only", "in 4-letter sessions")):
    w = (abs(sel[col] - sel["Supp. Fig. 2a"]) <= 0.1 * sel["Supp. Fig. 2a"])
    print(f"Rule '{rule}': within 10% of Supplementary Fig. 2a for", int(w.sum()), "of 10 mice;",
          "group totals", sel.groupby("group")[col].sum().to_dict())

**Result: partially reproduced.** The criterion says only when the claim is reproduced (both totals within tolerance). The outcome is partial because the claim holds in part: one of its two totals is within tolerance and the other is not. With the groups of Supplementary Fig. 2a (reading P), young APP-KI mice have 891 units, within 846 ± 85, and young WT mice 635, outside 479 ± 48. Principal cells number 677 and 564 against the paper's 636 and 423, interneurons (FS) 212 and 71 against 210 and 56. With 01398 and 01431 counted as APP-KI, as the file names label them (reading F), both totals fall outside (1164 and 362). Old and L-DOPA cohorts also hold more units than the paper reports (860 and 778 against 671 and 618; 263 and 286 against 144 and 132). The surplus is not explained by the odor sets: no single rule over sessions gives the counts of
Supplementary Fig. 2a. Counting all sessions puts 7 of 10 mice within 10%, and counting only the
sessions with a four-letter odor set 6 of 10. Mouse 01398's first four sessions, its four-letter
ones, hold exactly 112 principal cells, the paper's count, and its five doubled-letter sessions 91
more; mouse 00375's first five sessions hold 49 against 48. Mouse 00789's sessions all have
doubled-letter odor sets, and its 136 cells are near the paper's 125, so these sessions were not
all left out.

## ed4a-spike-type-distribution

> The proportions of narrow, regular and wide spiking LEC L2/3 cells differ between young WT and
> young APP-KI mice.

Blocked: the files hold no waveforms, so the NS, RS and WS width classes cannot be formed. As a
variant that does not decide the outcome, the two classes of the source file names (RS, FS) are
compared between the young groups.

In [ ]:
from scipy.stats import chi2_contingency
for reading in ("P", "F"):
    g = young_groups(reading)
    d = units[g.isin(["young WT", "young APP-KI"])].assign(g=g)
    tab = pd.crosstab(d.g, d.cls)
    chi_y = chi2_contingency(tab, correction=True); chi_n = chi2_contingency(tab, correction=False)
    print(f"Reading {reading}:"); print(tab.assign(**{"FS share": (tab.FS / tab.sum(axis=1)).round(3)}).to_string())
    print(f"  chi2 = {chi_n.statistic:.1f} (Yates {chi_y.statistic:.1f}), df 1, p = {chi_n.pvalue:.2g} (Yates {chi_y.pvalue:.2g})\n")
print("Paper (three classes): chi2 = 105.51, p = 1.2e-23.")

**Result: blocked, data missing.** The three width classes cannot be formed. In the two-class
variant, the FS share is larger in young APP-KI than in young WT mice under both groupings (chi2
and p printed above); this is not the claim's test.

## ed4c-higher-firing-rates

> Mean firing rates of putative principal neurons and interneurons in LEC L2/3 are higher in young
> APP-KI mice than in young WT mice.

Reading (a), the rate over the whole recording, is computed as the spike count over the
first-to-last spike span, with the source `meanFR` beside it; reading (b), the trial periods
only, cannot be computed, as the files hold no trial times. Principal cells are the RS-labelled
cells with more than 100 spikes, interneurons the FS-labelled cells (all spike counts, as
Extended Data Fig. 4 uses all neurons). Groups under readings P and F of units-recorded.

In [ ]:
rows = []
for reading in ("P", "F"):
    g = young_groups(reading)
    d = units.assign(g=g)
    for cls, sel in (("principal", (d.cls == "RS") & (d.n_spikes > 100)), ("interneuron", d.cls == "FS"),
                     ("principal, all spikes", d.cls == "RS")):
        for col in ("rate_span", "meanFR", "peak_rate"):
            a = d[sel & (d.g == "young WT")][col].dropna(); b = d[sel & (d.g == "young APP-KI")][col].dropna()
            z = stats.ranksums(a, b)
            rows.append({"groups": reading, "cells": cls, "measure": col, "n WT": len(a), "n APP-KI": len(b),
                         "median WT": round(a.median(), 3), "median APP-KI": round(b.median(), 3),
                         "mean WT": round(a.mean(), 3), "mean APP-KI": round(b.mean(), 3),
                         "Z (WT - APP)": round(z.statistic, 2), "p": z.pvalue,
                         "APP-KI higher at p<0.05": bool(z.pvalue < 0.05 and b.median() > a.median())})
fr = pd.DataFrame(rows)
print(fr.to_string(index=False))
print("\nPaper: principal Z = -7.85, p = 3.9e-15; interneurons Z = -2.84, p = 0.0044; principal peak rate Z = -4.06, p = 4.7e-5.")
print("Range of rate_span over young cells (Hz):", round(units[units.group.str.startswith("young", na=False)].rate_span.min(), 3), "to",
      round(units[units.group.str.startswith("young", na=False)].rate_span.max(), 2))

In [ ]:
pm_rows = []
d = units[units.group.isin(["young WT", "young APP-KI"])]
for cls, sel in (("principal", (d.cls == "RS") & (d.n_spikes > 100)), ("interneuron", d.cls == "FS")):
    per = d[sel].groupby(["group", "mouse"]).rate_span.agg(["mean", "median", "count"])
    print(cls, "per mouse (Hz):"); print(per.round(3).to_string())
    a = per.loc["young WT"]["mean"]; b = per.loc["young APP-KI"]["mean"]
    t = stats.ttest_ind(a, b)
    print(f"  per mouse mean rate, unpaired t-test: t({len(a) + len(b) - 2}) = {t.statistic:.2f}, p = {t.pvalue:.3g}\n")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for i, (cls, sel) in enumerate((("principal", (d.cls == "RS") & (d.n_spikes > 100)), ("interneuron", d.cls == "FS"))):
    for g, c in (("young WT", "tab:blue"), ("young APP-KI", "tab:red")):
        v = np.sort(d[sel & (d.group == g)].rate_span.values)
        ax[i].plot(v, np.linspace(0, 1, len(v)), color=c, label=g)
    ax[i].set_xscale("log"); ax[i].set_xlabel("rate over the recording (Hz)"); ax[i].set_title(cls); ax[i].legend()
plt.tight_layout(); plt.show()
print("All plotted values are inside the axes (log scale over each group's range).")

**Result: reproduced.** Over the recorded span, principal (RS) cells fire more in young APP-KI mice (median 1.22 against 0.82 Hz, 677 and 564 cells, Z = -7.18, p = 6.8e-13), and so do interneurons (FS; 1.13 against 0.89 Hz, 212 and 71 cells, Z = -2.21, p = 0.027). The source meanFR gives the same (p = 1.1e-12 and 0.031), as do the file labels' groups (reading F; p = 6.8e-16 and 0.0073). Per mouse, the principal rate is higher in APP-KI (t(8) = -4.84, p = 0.0013), the interneuron rate not significantly (t(8) = -1.60, p = 0.149). The principal peak rate is higher in APP-KI (28.7 against 27.1 Hz, p = 0.0066). Reading (b), trial periods only, cannot be computed.

## Behaviour of the photometry mice

The dandiset holds behaviour (odor and action per trial) only for the photometry mice: 5 DAT-Cre
(WT) and 5 APP-KI x DAT-Cre mice, which Supplementary Table 2 lists under Fig. 1. The tetrode
mice and the behaviour-only mice have none. The claims on young mice are tested on these 10 of
the paper's 11 WT and 16 APP-KI mice.

In [ ]:
beh = lecda.behaviour_table(sessions)
wide = beh.pivot_table(index=["strain", "mouse", "session", "n_trials"], columns="reading", values=["AB", "12"]).round(1)
print(wide.to_string())
print("\nSessions where trials 121 to 160 hold no Odor-1/2 trials (B1 undefined):", beh[(beh.reading == "B1") & beh["12"].isna()].session.tolist())

## fig1d-young-new-association-impaired

> Young APP-KI mice perform worse than young WT mice on novel odor associations at T5 (78.6%
> against 90.6% correct).

In [ ]:
def fig1d_test(reading):
    b = beh[beh.reading == reading]
    m = b.groupby(["strain", "mouse"])[["AB", "12"]].mean().reset_index()
    long = m.melt(id_vars=["strain", "mouse"], value_vars=["AB", "12"], var_name="pair", value_name="y")
    table, tk = lecda.two_way(long, "pair", "strain")
    return m, long, table, tk
rows = []
for reading in ("B1", "B2"):
    m, long, table, tk = fig1d_test(reading)
    print(f"Reading {reading}: per mouse T5 percent correct")
    print(m.round(1).to_string(index=False))
    print(m.groupby("strain")[["AB", "12"]].agg(["mean", "sem"]).round(2).to_string())
    print(table.round(4).to_string())
    print(tk.round(4).to_string(index=False))
    t12 = tk[(tk.a.map(lambda x: x[0]) == "12") & (tk.b.map(lambda x: x[0]) == "12")].iloc[0]
    tab_ = tk[(tk.a.map(lambda x: x[0]) == "AB") & (tk.b.map(lambda x: x[0]) == "AB")].iloc[0]
    means = m.groupby("strain")["12"].mean()
    rows.append({"reading": reading, "WT 1/2": round(means["WT"], 1), "APP-KI 1/2": round(means["APP-KI"], 1),
                 "WT within 90.6+-0.9": abs(means["WT"] - 90.6) <= 0.9, "APP within 78.6+-2.1": abs(means["APP-KI"] - 78.6) <= 2.1,
                 "interaction p": table.loc["C(pair):C(strain)", "PR(>F)"], "Tukey 1/2 p": t12.p, "Tukey A/B p": tab_.p})
    print()
print(pd.DataFrame(rows).to_string(index=False))
print("Paper: F(1,50) = 20.2, p = 4.1e-5 (interaction); Tukey 1/2 p = 3.1e-8; A/B p = 0.92 (young).")

In [ ]:
from statsmodels.stats.power import TTestIndPower
pw = TTestIndPower()
sd_wt, sd_app = 0.9 * np.sqrt(11), 2.1 * np.sqrt(16)
sd = np.sqrt((10 * sd_wt ** 2 + 15 * sd_app ** 2) / 25)
d_eff = (90.6 - 78.6) / sd
print(f"Paper's 1/2 T5 effect: SD {sd_wt:.1f} and {sd_app:.1f} (from s.e.m. and n), pooled {sd:.1f}, d = {d_eff:.2f}")
print(f"Power of a two-sided t-test at alpha 0.05 for this effect: 5 + 5 mice {pw.power(d_eff, 5, 0.05, ratio=1):.2f}; "
      f"11 + 16 mice {pw.power(d_eff, 11, 0.05, ratio=16 / 11):.2f}")
print("(The Tukey comparison over four cells is stricter than this t-test, so its power is lower.)")

**Result: not reproduced.** On the 10 photometry mice that have behaviour (5 WT and 5 APP-KI of the paper's 11 and 16), APP-KI mice are at 77.8% (B1) or 79.9% (B2) correct on Odor-1/2 at T5, within 78.6 ± 2.1, and WT mice at 87.3% or 88.8%, below 90.6 ± 0.9. Neither the odor pair by strain interaction (p = 0.26 and 0.13) nor the 1/2 Tukey comparison (p = 0.36 and 0.21) is significant. The A/B pair does not differ between strains (Tukey p = 0.9999 and 0.997). A failed test here is weak evidence: with the paper's own effect (d = 1.77 for the 1/2 comparison),
a t-test on 5 against 5 mice has power 0.69, against 0.99 for the paper's 11 and 16 mice, and the
Tukey comparison has less.

## fig1e-young-correct-sessions

> Young WT mice learn the new association in 77.5% of sessions, young APP-KI mice in 59.1%.

In [ ]:
rows = []
for reading in ("B1", "B2"):
    for cut in (">", ">="):
        b = beh[(beh.reading == reading) & beh["12"].notna()].copy()
        b["correct"] = (b["12"] > 80) if cut == ">" else (b["12"] >= 80)
        per = b.groupby(["strain", "mouse"]).correct.mean().mul(100)
        a, c = per["WT"], per["APP-KI"]
        t = stats.ttest_ind(a, c)
        rows.append({"reading": reading, "cut": cut, "WT mean": round(a.mean(), 1), "WT sem": round(a.sem(), 1),
                     "APP mean": round(c.mean(), 1), "APP sem": round(c.sem(), 1),
                     "WT within 77.5+-6.1": abs(a.mean() - 77.5) <= 6.1, "APP within 59.1+-5.5": abs(c.mean() - 59.1) <= 5.5,
                     "t": round(t.statistic, 2), "df": len(a) + len(c) - 2, "p": round(t.pvalue, 4),
                     "per mouse WT": a.round(0).tolist(), "per mouse APP": c.round(0).tolist()})
print(pd.DataFrame(rows).to_string(index=False))
print("Sessions left out, T5 performance undefined:", beh[beh["12"].isna()][["session", "reading"]].values.tolist())
print("Sessions at exactly 80%:", beh[beh["12"] == 80][["session", "reading"]].values.tolist())
print("Paper: t(25) = 2.19, p = 0.037.")

In [ ]:
sd_wt, sd_app = 6.1 * np.sqrt(11), 5.5 * np.sqrt(16)
sd = np.sqrt((10 * sd_wt ** 2 + 15 * sd_app ** 2) / 25)
d_eff = (77.5 - 59.1) / sd
print(f"Paper's correct-session effect: pooled SD {sd:.1f}, d = {d_eff:.2f}")
print(f"Power of the two-sided t-test at alpha 0.05: 5 + 5 mice {pw.power(d_eff, 5, 0.05, ratio=1):.2f}; "
      f"11 + 16 mice {pw.power(d_eff, 11, 0.05, ratio=16 / 11):.2f}")

**Result: partially reproduced.** WT mice have more correct sessions than APP-KI mice under both readings. Under B1, with session 00266_20190405 left out (its T5 performance is undefined), both means are within tolerance (83.3% against 77.5 ± 6.1; 57.2% against 59.1 ± 5.5), but the difference is not significant (t(8) = 2.12, p = 0.067). Under B2 it is significant (t(8) = 2.36, p = 0.046) and APP-KI is within tolerance (61.9%), but WT is above it (90.0%). No reading meets every part of the criterion. The criterion says only when the claim is reproduced; the outcome is partial because the claim holds in part: the direction holds under both readings, the two means under B1 and the test under B2. No session is at exactly 80%, so the two cut-offs agree. The test has little power on 10 mice: for the paper's effect (d = 0.86), power is 0.23 with 5 and 5 mice, against 0.56 with the paper's 11 and 16.

## Photometry: common table

Per session, odor and timepoint, the window means of the block's trial-averaged z-scored signal
(helper, `block_table`). Units of analysis: the paper's are hemispheres (10 WT, 8 APP-KI), which
the files do not identify (one signal per session file, no fiber or location metadata; data
check). The Fig. 4d legend gives F(4,80) for 10 + 8 hemispheres and five timepoints, which leaves
one value per hemisphere and timepoint: a hemisphere's sessions are pooled, not counted apart.
The adopted reading, an assumption, is therefore the mean over a mouse's sessions (M; 5 WT and 5
APP-KI mice), the nearest pooled unit the files allow. It has fewer units than the paper's 18,
and with 5 units a two-sided signed-rank test cannot reach p < 0.05 (its smallest p is 0.0625).
Sessions as units (S; 11 WT and 30 APP-KI sessions) are run as a variant: they count repeated
recordings of one hemisphere as independent, giving 195 residual degrees of freedom against the
paper's 80. Where the two readings give different outcomes, this is said in the conclusion.

Against the precue, the Wilcoxon signed-rank test is adopted (Supplementary Table 1). It is
paired; the rank sum test the Results name compares window means with precue means that are near
0 by the z-scoring, so it is not paired, and it is shown as a variant. A session whose T5
performance is undefined under a reading (00266_20190405 under B1) is counted neither as correct
nor as an error.

In [ ]:
blocks = lecda.block_table(sessions)
print("Rows:", len(blocks), "; sessions:", blocks.session.nunique(), "; trials per T1-T5 block:", sorted(blocks[blocks["T"].str.len() == 2].n_trials.unique()))
print("Range of window means (z):", {w: (round(blocks[w].min(), 2), round(blocks[w].max(), 2)) for w in lecda.WINDOWS})
print("Range of precue means (z):", round(blocks.pre.min(), 3), "to", round(blocks.pre.max(), 3))
for reading in ("B1", "B2"):
    for cut in (">", ">="):
        c = lecda.is_correct(blocks.drop_duplicates("session"), reading, cut)
        e = lecda.is_error(blocks.drop_duplicates("session"), reading, cut)
        print(f"Correct sessions, {reading} {cut}80%:", blocks.drop_duplicates("session")[c].groupby("strain").size().to_dict(),
              "error:", blocks.drop_duplicates("session")[e].groupby("strain").size().to_dict(),
              "undefined:", blocks.drop_duplicates("session")[~c & ~e].session.tolist())
TS = ["T1", "T2", "T3", "T4", "T5"]
def precue_tests(df):
    """Window G2 against the precue per unit: Wilcoxon signed-rank (paired, the test adopted)
    and rank sum (a variant; it compares window means with precue means that are near 0 by
    the z-scoring, so it is not a paired test)."""
    if len(df) < 2:
        return np.nan, np.nan
    sr = stats.wilcoxon(df.G2, df.pre).pvalue if len(df) > 1 else np.nan
    rs = stats.ranksums(df.G2, df.pre).pvalue
    return sr, rs

## fig4b-wt-dopamine-t1

> In correct sessions of control mice, LEC dopamine fibers respond at T1 to prelearned Odor-A, to
> novel Odor-1 and to novel Odor-2; with learning the Odor-2 signal disappears while the Odor-A and
> Odor-1 signals remain to T5.

In [ ]:
exp = {"A": (1.1, 0.24), "1": (1.8, 0.2), "2": (0.76, 0.24)}
rows = []
for unit in ("M", "S"):
    for reading in ("B1", "B2"):
        for cut in (">", ">="):
            c = blocks[(blocks.strain == "WT") & lecda.is_correct(blocks, reading, cut)]
            u = lecda.per_unit(c, "session" if unit == "S" else "mouse")
            r = {"unit": unit, "reading": reading, "cut": cut, "n": u.unit.nunique()}
            for w in ("G1", "G2", "G4"):
                ok = True
                for od in ("A", "1", "2"):
                    v = u[(u["T"] == "T1") & (u.odor == od)][w].mean()
                    r[f"{w} {od}"] = round(v, 2)
                    ok &= abs(v - exp[od][0]) <= exp[od][1]
                r[f"{w} all in"] = ok
            for T in ("T1", "T5"):
                for od in ("A", "1", "2"):
                    sr, rs = precue_tests(u[(u["T"] == T) & (u.odor == od)])
                    r[f"{T} {od} sr"] = round(sr, 4); r[f"{T} {od} rs"] = round(rs, 4)
            rows.append(r)
f4b = pd.DataFrame(rows)
print(f4b[["unit", "reading", "cut", "n"] + [c for c in f4b.columns if c.startswith(("G1", "G2", "G4"))]].to_string(index=False))
print()
print(f4b[["unit", "reading", "cut", "n"] + [c for c in f4b.columns if c.startswith(("T1", "T5"))]].to_string(index=False))
for test in ("sr", "rs"):
    f4b[f"tests hold ({test})"] = ((f4b[[f"T1 {o} {test}" for o in "A12"]] < 0.05).all(axis=1) & (f4b[f"T5 A {test}"] < 0.05)
                                   & (f4b[f"T5 1 {test}"] < 0.05) & (f4b[f"T5 2 {test}"] >= 0.05))
f4b["means in (any window)"] = f4b[["G1 all in", "G2 all in", "G4 all in"]].any(axis=1)
print()
print(f4b[["unit", "reading", "cut", "n", "means in (any window)", "tests hold (sr)", "tests hold (rs)"]].to_string(index=False))
print("Paper: T1 Odor-A 1.1 +- 0.24, Odor-1 1.8 +- 0.2, Odor-2 0.76 +- 0.24 (n = 10 hemispheres); p = 6.9e-3, 5.1e-3, 0.028 against precue.")

In [ ]:
c = blocks[(blocks.strain == "WT") & lecda.is_correct(blocks, "B2", ">")]
per_mouse = c[c.odor.isin(["A", "1", "2"]) & c["T"].isin(["T1", "T5"])].pivot_table(index="mouse", columns=["T", "odor"], values="G2").round(2)
print("Per mouse, correct sessions (B2), G2 window means:"); print(per_mouse.to_string())

**Result: partially reproduced, resting on the assumed timing and unit of analysis.** With mice as units (the adopted reading, 5 WT mice), the T1 means are within tolerance in the 1 to 4 s window (B1: Odor-A 1.17, Odor-1 1.66, Odor-2 0.55; B2: 1.11, 1.63, 0.55) and in the 2 to 4 s window, but the significance part is settled by the choice of unit, not by the data: with 5 mice the smallest possible two-sided signed-rank p is 0.0625, so no test can pass, and this part is not tested on this unit (by rank sum Odor-2 at T1 has p = 0.117). The outcome stands because the sessions variant, where the test can be run, gives the same outcome. With sessions as units (variant) and B2 (10 correct sessions), the 2 to 4 s means are 1.29, 1.67 and 0.55, all within tolerance; all three are above the precue at T1 by signed-rank (p = 0.0020, 0.0039, 0.049); at T5 Odor-A is significant (p = 0.0020) and Odor-2 is not (p = 0.13), but Odor-1 is not significant (p = 0.084), so that part fails. By rank sum, a variant that is not paired, every condition holds (Odor-1 at T5 p = 0.0025). Under B1 with sessions the Odor-2 T1 mean falls below tolerance (0.48 at 2 to 4 s). The means hold under both units; the tests hold in part with sessions, and with mice they are ruled out by the design.

## fig4c-app-dopamine-delayed

> In APP-KI mice, Odor-1 dopamine signals emerge late, from T3, in correct sessions and are absent
> in error sessions (39% of sessions), while Odor-A signals stay intact.

The two denominators of the criterion coincide here, since each file is one session of one mouse.

In [ ]:
sess = blocks.drop_duplicates("session")
app_s = sess[sess.strain == "APP-KI"]
rows = []
for reading in ("B1", "B2"):
    for cut in (">", ">="):
        e = lecda.is_error(app_s, reading, cut)
        rows.append({"reading": reading, "cut": cut, "APP-KI sessions": len(app_s), "error": int(e.sum()),
                     "error %": round(100 * e.mean(), 1), "within 39+-3.9": abs(100 * e.mean() - 39) <= 3.9})
print(pd.DataFrame(rows).to_string(index=False))
rows = []
for unit in ("M", "S"):
    for reading in ("B1", "B2"):
        for cut in (">", ">="):
            a = blocks[blocks.strain == "APP-KI"]
            corr = lecda.is_correct(a, reading, cut)
            for kind, sel in (("correct", corr), ("error", lecda.is_error(a, reading, cut))):
                u = lecda.per_unit(a[sel], "session" if unit == "S" else "mouse")
                r = {"unit": unit, "reading": reading, "cut": cut, "sessions": kind, "n": u.unit.nunique()}
                for od in ("1", "A"):
                    for T in TS:
                        r[f"{od} {T}"] = round(precue_tests(u[(u["T"] == T) & (u.odor == od)])[0], 4)
                rows.append(r)
f4c = pd.DataFrame(rows)
print("\nWilcoxon signed-rank p, G2 window against precue:")
print(f4c.to_string(index=False))
def f4c_ok(df):
    cr = df[df.sessions == "correct"].iloc[0]; er = df[df.sessions == "error"].iloc[0]
    return {"Odor-1 correct: ns T1,T2, sig T3": bool(cr["1 T1"] >= 0.05 and cr["1 T2"] >= 0.05 and cr["1 T3"] < 0.05),
            "Odor-1 error: ns at all": bool((er[[f"1 {T}" for T in TS]] >= 0.05).all()),
            "Odor-A sig T1,T5 both": bool(cr["A T1"] < 0.05 and cr["A T5"] < 0.05 and er["A T1"] < 0.05 and er["A T5"] < 0.05)}
summ = pd.DataFrame([{"unit": k[0], "reading": k[1], "cut": k[2], **f4c_ok(g)} for k, g in f4c.groupby(["unit", "reading", "cut"])])
print(summ.to_string(index=False))

In [ ]:
rs_rows = []
for unit in ("S",):
    for reading in ("B1", "B2"):
        a = blocks[blocks.strain == "APP-KI"]
        corr = lecda.is_correct(a, reading, ">")
        for kind, sel in (("correct", corr), ("error", lecda.is_error(a, reading, ">"))):
            u = lecda.per_unit(a[sel], "session")
            rs_rows.append({"reading": reading, "sessions": kind, **{f"1 {T}": round(precue_tests(u[(u["T"] == T) & (u.odor == "1")])[1], 4) for T in TS},
                            **{f"1 {T} mean": round(u[(u["T"] == T) & (u.odor == "1")].G2.mean(), 2) for T in TS}})
print("Variant: rank sum p and mean G2 for Odor-1, sessions as units:")
print(pd.DataFrame(rs_rows).to_string(index=False))

**Result: not reproduced.** Error sessions are 40.0% of the 30 APP-KI sessions under B2, within 39 ± 3.9 (43.3% under B1). With mice as units (adopted), the significance parts are settled by the choice of unit, not by the data: with 5 mice the smallest possible two-sided signed-rank p is 0.0625, so the Odor-1 signal is never significant in correct sessions and Odor-A is not significant at T1 or T5 in either kind of session, whatever the data. The outcome stands because the sessions variant, where the tests can be run, also fails, on the parts listed next. With sessions as units (variant), the Odor-1 signal in correct sessions is already significant at T1 under B2 (p = 0.016) and at T2 under B1 (p = 0.0005), not first at T3; in error sessions it is significant at T4 (p = 0.021 under B2, 0.011 under B1); and Odor-A in error sessions is not significant at T1 (p = 0.42 and 0.15). In correct sessions the Odor-1 mean does grow, from 0.44 at T1 to 0.91, 1.09, 1.15 and 0.96 (B2), and in error sessions it stays between -0.02 and 0.32.

## fig4d-app-odor1-dopamine-decreased

> Across all sessions, LEC dopamine activity during Odor-1 trials is significantly lower in APP-KI
> mice than in WT mice at T1.

In [ ]:
def strain_time(df, odor, window, unit):
    u = lecda.per_unit(df, "session" if unit == "S" else "mouse")
    v = u[(u.odor == odor) & u["T"].isin(TS)].rename(columns={window: "y"})
    table, tk = lecda.two_way(v, "strain", "T")
    same = tk[[a[1] == b[1] for a, b in zip(tk.a, tk.b)]].copy()
    same["T"] = [a[1] for a in same.a]
    # diff is a - b; report WT minus APP-KI
    same["WT - APP"] = [d if a[0] == "WT" else -d for a, d in zip(same.a, same["diff"])]
    n = v.groupby("strain").unit.nunique().to_dict()
    return table, same, n
def st_row(df, odor, window, unit, label):
    table, same, n = strain_time(df, odor, window, unit)
    t1 = same[same["T"] == "T1"].iloc[0]
    return {"subset": label, "odor": odor, "window": window, "unit": unit, "n": n,
            "F int": round(table.loc["C(strain):C(T)", "F"], 2), "df": (int(table.loc["C(strain):C(T)", "df"]), int(table.loc["Residual", "df"])),
            "p int": round(table.loc["C(strain):C(T)", "PR(>F)"], 4), "p strain": round(table.loc["C(strain)", "PR(>F)"], 4),
            "T1 WT-APP": round(t1["WT - APP"], 2), "T1 Tukey p": round(t1.p, 4),
            "min same-T Tukey p": round(same.p.min(), 4), "at": same.loc[same.p.idxmin(), "T"]}
rows = [st_row(blocks, "1", w, unit, "all") for unit in ("M", "S") for w in ("G1", "G2", "G3")]
f4d = pd.DataFrame(rows); print(f4d.to_string(index=False))
print("Paper: F(4,80) = 2.5, p = 0.047; T1 Tukey p = 4.2e-3 (10 and 8 hemispheres).")
m = blocks[(blocks.odor == "1") & blocks["T"].isin(TS)].groupby(["strain", "mouse", "T"]).G1.mean().unstack().round(2)
print("\nPer mouse Odor-1 G1 means (all sessions):"); print(m.to_string())

**Result: not reproduced.** In the 1 to 4 s window the criterion fixes, the Odor-1 signal at T1 is lower in APP-KI than in WT mice by 0.94 z with mice as units (adopted) and 0.95 with sessions. Neither the strain by time interaction (mice F(4,40) = 1.72, p = 0.16; sessions F(4,195) = 1.40, p = 0.23) nor the T1 Tukey comparison (mice p = 0.059; sessions p = 0.17) is significant. Per mouse, each APP-KI T1 value (0.17 to 0.81) is below each WT value (0.85 to 1.74). In the 2 to 4 s window, a variant, mice give interaction p = 0.091 and T1 Tukey p = 0.0032, and sessions p = 0.042 and 0.0020.

## fig4e-correct-sessions-odor1-decreased

> In correct sessions, Odor-1 dopamine activity at T1 is also lower in APP-KI mice than in WT
> mice, with no significant interaction for Odor-2.

In [ ]:
rows = []
for unit in ("M", "S"):
    for reading in ("B1", "B2"):
        for cut in (">", ">="):
            c = blocks[lecda.is_correct(blocks, reading, cut)]
            for od in ("1", "2"):
                rows.append(st_row(c, od, "G1", unit, f"correct {reading} {cut}"))
print(pd.DataFrame(rows).to_string(index=False))
print("Paper: Odor-1 F(4,80) = 3.72, p = 7.8e-3; T1 Tukey p = 4.8e-3; Odor-2 F(4,80) = 2.1, p = 0.093.")

**Result: not reproduced.** In correct sessions, Odor-1 at T1 is lower in APP-KI mice by 0.86 to 0.95 z. Under no reading (B1, B2; mice, sessions) are the strain by time interaction (p = 0.08 to 0.33) or the T1 Tukey comparison (p = 0.33 to 0.68) significant. As the paper reports, the Odor-2 interaction is not significant (p = 0.62 to 0.92).

## fig4f-error-sessions-no-difference

> In error sessions, dopamine activity does not differ between young WT and young APP-KI mice.

In [ ]:
rows = []
for unit in ("M", "S"):
    for reading in ("B1", "B2"):
        for cut in (">", ">="):
            e = blocks[lecda.is_error(blocks, reading, cut)]
            for od in ("A", "B", "1", "2"):
                rows.append(st_row(e, od, "G1", unit, f"error {reading} {cut}"))
f4f = pd.DataFrame(rows)
f4f["absent"] = (f4f["p int"] >= 0.05) & (f4f["min same-T Tukey p"] >= 0.05)
print(f4f.to_string(index=False))
print("WT error sessions:", blocks.drop_duplicates("session").pipe(lambda s: s[(s.strain == "WT")]).assign(
    B1=lambda s: lecda.is_error(s, "B1", ">"), B2=lambda s: lecda.is_error(s, "B2", ">"))[["session", "B1", "B2"]].query("B1 or B2").to_string(index=False))

**Result: reproduced.** In error sessions, under every reading, no odor shows a significant strain by time interaction (p = 0.22 to 0.998) or a significant WT against APP-KI Tukey comparison at the same timepoint (smallest p = 0.17). The test has little power: there are 1 (B2) or 2 (B1) WT error sessions, from 1 or 2 mice. A strain main effect (type III) appears for Odor-1 under B1 with mice as units (p = 0.011; sessions p = 0.063), as Supplementary Table 1 reports strain effects for these data.

## figs7-odor1-specific-decrease

> The WT minus APP-KI dopamine difference at T1 is larger for Odor-1 than for Odor-A or Odor-B.

Each APP-KI unit's T1 value is subtracted from the WT mean of the same odor (claims file), window
G1, all sessions.

In [ ]:
rows = []
for unit in ("M", "S"):
    u = lecda.per_unit(blocks[blocks["T"] == "T1"], "session" if unit == "S" else "mouse")
    wt = u[u.strain == "WT"].groupby("odor").G1.mean()
    app = u[u.strain == "APP-KI"].copy(); app["d"] = app.odor.map(wt) - app.G1
    groups = [app[app.odor == o].d.values for o in ("A", "B", "1", "2")]
    f = stats.f_oneway(*groups); tk = stats.tukey_hsd(*groups)
    rows.append({"unit": unit, "n per odor": len(groups[0]), **{f"mean d {o}": round(g.mean(), 2) for o, g in zip("AB12", groups)},
                 "F": round(f.statistic, 2), "df": (3, 4 * len(groups[0]) - 4), "p": round(f.pvalue, 5),
                 "Tukey A vs 1": round(tk.pvalue[0, 2], 4), "Tukey B vs 1": round(tk.pvalue[1, 2], 4),
                 "Tukey 2 vs 1": round(tk.pvalue[3, 2], 4)})
print(pd.DataFrame(rows).to_string(index=False))
print("Paper: F(3,28) = 3.98, p = 0.017; Tukey A vs 1 p = 0.042, B vs 1 p = 0.041.")

**Result: partially reproduced, resting on the unit of analysis.** With mice as units (adopted, 5 per odor), the WT mean minus each APP-KI mouse at T1 is 0.94 z for Odor-1, against -0.25 for Odor-A, 0.15 for Odor-B and 0.24 for Odor-2; F(3,16) = 6.24, p = 0.0052. Odor-1 exceeds Odor-A (Tukey p = 0.0031) but not Odor-B significantly (p = 0.055), so one of the two comparisons fails. With sessions as units (variant, 30 per odor), the differences are 0.95, 0.18, 0.15 and 0.36, F(3,116) = 6.70, p = 0.00033, and Odor-1 exceeds both Odor-A (p = 0.0013) and Odor-B (p = 0.0008). The outcome depends on the unit: partial under mice, reproduced under sessions. Neither is the paper's hemisphere unit; the outcome follows the adopted mouse unit.

## figs6d-dopamine-behavior-correlation

> In APP-KI mice, the trial at which Odor-1 dopamine signals emerge correlates with the trial at
> which licking to Odor-1 is learned.

Per APP-KI session: the first Odor-1 trial of the first run of five consecutive Odor-1 hits,
against the first of the first run of five consecutive Odor-1 trials whose signal is above the
3 SD threshold. Readings of the claims file: the SD of the precue (a) or of the session's signal
(b), windows G1 and G2. Two details the claims file does not give are run as readings too: the
signal is the window mean (m) or the window's maximum (x), and trials are counted among Odor-1
trials (o) or among all trials of the session (s). The threshold is the mean plus 3 SD of the
reference.

In [ ]:
rows = []
app_sessions = [s for s in sessions if s["strain"] == "APP-KI"]
for sd in ("a", "b"):
    for w in ("G1", "G2"):
        for meas in ("m", "x"):
            for count in ("o", "s"):
                xs, ys, dropped = [], [], 0
                for s in app_sessions:
                    tr = np.flatnonzero(s["odor"] == 3)
                    sig = s["Xs"][tr][:, lecda.window_slice(*lecda.WINDOWS[w])]
                    v = sig.mean(1) if meas == "m" else sig.max(1)
                    if sd == "a":
                        ref = s["Xs"][tr][:, lecda.window_slice(*lecda.PRECUE)]
                    else:
                        ref = s["Xs"]
                    thr = ref.mean() + 3 * ref.std()
                    b = lecda.first_run(s["action"][tr] == lecda.HIT)
                    g = lecda.first_run(v > thr)
                    if np.isnan(b) or np.isnan(g):
                        dropped += 1; continue
                    if count == "s":
                        b, g = tr[int(b)], tr[int(g)]
                    xs.append(b); ys.append(g)
                r = stats.pearsonr(xs, ys) if len(xs) > 2 else None
                rows.append({"SD": sd, "window": w, "signal": meas, "count": count, "sessions": len(xs), "dropped": dropped,
                             "r": round(r.statistic, 3) if r else np.nan, "p": round(r.pvalue, 4) if r else np.nan,
                             "r within 0.78+-0.078": bool(r and abs(r.statistic - 0.78) <= 0.078),
                             "p<0.05": bool(r and r.pvalue < 0.05)})
s6 = pd.DataFrame(rows)
print(s6.to_string(index=False))
print("APP-KI sessions:", len(app_sessions), "; paper: r(19) = 0.781, p = 2.5e-5, 21 sessions.")

In [ ]:
rows = []
for s in app_sessions:
    tr = np.flatnonzero(s["odor"] == 3)
    pre = s["Xs"][tr][:, lecda.window_slice(*lecda.PRECUE)]
    row = {"session": f"{s['mouse']}_{s['date']}", "thr (a)": pre.mean() + 3 * pre.std(), "thr (b)": s["Xs"].mean() + 3 * s["Xs"].std()}
    for w in ("G1", "G2"):
        sig = s["Xs"][tr][:, lecda.window_slice(*lecda.WINDOWS[w])]
        row[f"{w} mean, 90th pct over Odor-1 trials"] = np.percentile(sig.mean(1), 90)
        row[f"{w} max, 90th pct"] = np.percentile(sig.max(1), 90)
    rows.append(row)
th = pd.DataFrame(rows)
print("Thresholds (mean + 3 SD) and Odor-1 signal levels per APP-KI session, medians over the 30 sessions:")
print(th.drop(columns="session").median().round(2).to_string())
print("Sessions whose 90th percentile G1 window mean exceeds threshold (b):", int((th["G1 mean, 90th pct over Odor-1 trials"] > th["thr (b)"]).sum()),
      "; G1 window max:", int((th["G1 max, 90th pct"] > th["thr (b)"]).sum()))

**Result: blocked, methods underspecified.** The readings disagree. With the precue SD (a), the correlation is positive in all 8 combinations and has p < 0.05 in 6 (r = 0.30 to 0.73, on 9 to 22 of 30 sessions). It is within 0.78 ± 0.078 only for the 2 to 4 s window mean counted among Odor-1 trials (r = 0.73, p = 0.025, 9 sessions). With the session's SD (b), at most 1 session has five consecutive Odor-1 trials above threshold, so no r can be computed. Reading (b) takes the SD over every sample of every trial, post-cue responses included, so its threshold (median 4.51 z over sessions) lies above the 90th percentile of the Odor-1 means in the 1 to 4 s window in every session (median 2.07 z). The window maximum passes it at the 90th percentile in 17 sessions, but five consecutive trials above it occur in at most 1. The window-mean readings under (a) keep 9 to 11 of 30 sessions, against the paper's 21.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| link-unit-mice | partially reproduced |
| link-photometry-mice | partially reproduced |
| units-recorded | partially reproduced |
| ed4c-higher-firing-rates | reproduced |
| fig1d-young-new-association-impaired | not reproduced (10 of 27 mice; assumed) |
| fig1e-young-correct-sessions | partially reproduced (10 of 27 mice; assumed) |
| fig4b-wt-dopamine-t1 | partially reproduced (assumed) |
| fig4c-app-dopamine-delayed | not reproduced (assumed) |
| fig4d-app-odor1-dopamine-decreased | not reproduced (assumed) |
| fig4e-correct-sessions-odor1-decreased | not reproduced (assumed) |
| fig4f-error-sessions-no-difference | reproduced (assumed) |
| figs7-odor1-specific-decrease | partially reproduced (assumed) |
| figs6d-dopamine-behavior-correlation | blocked: methods underspecified |
| ed4a, ed4b | blocked: data missing (no waveforms; a two-class RS/FS variant is shown for ed4a) |
| 13 claims on cue responses, PCA, similarity, decoding (Fig. 2, Extended Data Figs. 5 to 9, Fig. 6e to i) | blocked: data missing (no event times) |
| 4 histology claims | blocked: data missing |
| 13 behaviour claims on old, prelearning, optogenetic and L-DOPA mice | blocked: data missing |

How the analysis differs from the paper's. Cell classes are the files' RS and FS labels, not
spike widths. Firing rates are over the recorded span, not over trial periods. The photometry
traces are read as 1 kHz with the cue at sample 1000, from their z-scoring. Their units are
the mean over a mouse's sessions, the nearest the files allow to the paper's pooled hemispheres
(assumed), with sessions as a variant; with 5 mice a signed-rank test cannot reach p < 0.05, so those parts of fig4b and fig4c are settled by the unit, and their outcomes stand because the sessions variant gives the same ones. The behaviour and photometry claims
on young mice rest on the 5 WT and 5 APP-KI photometry mice only. The groups come from
Supplementary Table 2 and Supplementary Fig. 2a.

What was found about the dataset. The unit files hold no trial or event times, so no unit
can be aligned to an odor, and they hold no waveforms. The source file names label two young WT
mice of the paper (01398, 01431) as AD. Mouse 00375's 99 units are deposited twice (100 "_Sum"
copies with genotype "Sim1"). One unit file holds no units table. No file states genotype, age
group or L-DOPA treatment; Supplementary Table 2 swaps the young and old APP-KI tetrode mice
relative to Supplementary Fig. 2a and the files' birth dates. Unit counts exceed the paper's in
every group. Three subjects' sex differs from Supplementary Table 2. The photometry files
state no sampling rate, cue time, odor or action names, and hold one trace per session, so the
paper's hemispheres cannot be identified. Mouse 00163 is missing. Sessions hold 99 to 269 trials
and no separate A/B block, where the paper describes about 20 A/B trials and 160 AB12 trials.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")